<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 04 · 一次提问究竟需要多少上下文

项目推进了几周，已经积累了金额处理、坏行提示、发布流程和文档风格等约定。
此刻的问题只是“怎样处理 amount 字段”。把全部历史都塞给模型，会让简单的问题背上越来越多的上下文。

这一篇没有模型回答。我们直接检查模型调用之前的那份材料：它选择了什么，占了多少字节，又保留了哪些引用。

**这一篇的收获：** 准备有界的 PreparedContext，比较 query 与字节预算，观察空结果，并构造一次真实应用可以发送的消息。

**运行准备：** 从 [教程入口](README.md) 安装依赖并启动 Jupyter。每篇都带有自己的数据，可以独立运行。
本篇不需要模型或 API Key。
建议先逐格运行，读完输出再继续；完整重跑时使用 **Restart Kernel & Run All**。

## 先准备一个自己的实验空间

下面的辅助代码只负责启动本地 Server、建立 Client 和整理输出。默认每次完整运行使用新的 SQLite 数据库；选择 OceanBase 时，使用专用测试库并为本次实验创建新的 Scope。
后端设置见 [README](README.md#使用-oceanbase-运行)。关键的写入、检索、审核与交接调用会直接写在后面的单元格里。


本篇通过 `remember_memory` 写入 Scope 的日常 Memory，使后续搜索和上下文准备读取同一份知识。独立制品的创建、版本管理与日常记忆的关系见 [接口选择](DESIGN.md#接口选择)。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))

if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("04", features=())
client = lab.client
assert client is not None

现在创建本篇的项目 Scope。`title` 是给人看的名称，真正用于调用的是 Server 返回的 `scope_id`。
你可以改变标题；不要自己根据标题或目录拼出一个 Scope ID。


In [ ]:
from powercontext.http import CreateScopeRequest

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 04",
        summary="第 04 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-04",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 准备相关和无关的知识

先写入几条比较详细的金额约定，再放入发布流程与文档规则。为了能明显观察预算变化，
这里的金额记录比前几篇长。每一条仍是可以单独理解、单独修改的主题。


In [ ]:
from powercontext.http import ListMemoryEntriesRequest, PrepareContextRequest, RememberMemoryRequest

rules = [
    ("amount storage", "订单金额以整数分存储；100 表示 1 元。转换时先使用 Decimal 读取输入，再检查小数位和范围。"),
    ("amount rounding", "超过两位小数的金额应报告校验错误，不能静默四舍五入；报错保留行号，不回显整行内容。"),
    ("amount bounds", "负数金额和超出允许范围的金额应单独报错；空白字段也应有清楚的错误原因。"),
    ("amount regression", "金额回归检查覆盖零值、两位小数、大金额、空白和非法文本，输入与预期输出放在同一张用例表。"),
    ("release", "发布前先检查 changelog，并确认迁移操作的维护窗口。"),
    ("documentation", "文档以短段落解释目的，每个示例说明预期输出。"),
]
for topic, description in rules:
    text = f"{topic}: {description}"
    if topic.startswith("amount"):
        text += " 验证时记录输入类别、预期行为和观察到的结果；只有检查实际通过，才记录为已验证。" * 3
    await client.remember_memory(
        RememberMemoryRequest(scope_id=scope_id, kind="constraint", text=text, reason="示例项目规范")
    )
entries = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
table([
    {"主题": entry.text.split(":", 1)[0], "正文 UTF-8 字节": len(entry.text.encode("utf-8"))}
    for entry in entries.entries
])

## 2. 为当前问题准备上下文

`prepare_context` 返回的是一次调用要使用的临时视图。`max_bytes` 限制输出的 UTF-8 字节数，
不是 Python 字符数，也不是模型 token 数。请直接阅读下面的内容，找出约定与精确引用。


基础配置按 FTS 主题词 `amount` 选取材料，完整自然语言问题保留给应用模型；这有助于将主题选择与模型回答分开观察。

In [ ]:
import json

question = "amount 字段应该怎样解析和验证？"
prepared = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount", max_bytes=2500))
assert prepared.status == "ready"
assert prepared.content is not None
assert prepared.content_bytes == len(prepared.content.encode("utf-8"))
assert prepared.content_bytes <= 2500


def context_items(content):
    start = content.index("BEGIN_POWERCONTEXT_PREPARED_CONTEXT_V1") + len("BEGIN_POWERCONTEXT_PREPARED_CONTEXT_V1")
    end = content.index("END_POWERCONTEXT_PREPARED_CONTEXT_V1")
    return json.loads(content[start:end].strip())["items"]


items = context_items(prepared.content)
assert items, "query=amount 时应选中至少一条金额约定。"
assert all(item.get("citation") for item in items)
assert any("amount" in item["content"].lower() or "整数分" in item["content"] for item in items)
table([{"引用": item["citation"], "正文预览": item["content"][:96]} for item in items])
print(prepared.content)

## 3. 给相同问题不同的预算

先预测：预算变小时，是内容会被无限截断，还是最终可以提供的材料会减少？
下面只比较实际状态与字节数，不假定所有预算都能容纳一条完整记录。小预算下得到 `empty` 也是需要处理的结果。


In [ ]:
budgets = [512, 1024, 2500, 6000]
comparisons = []
for budget in budgets:
    view = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount", max_bytes=budget))
    assert view.content_bytes <= budget
    comparisons.append({"预算": budget, "状态": view.status, "实际字节": view.content_bytes})
table(comparisons)

## 4. 改变问题，观察材料变化

接下来问发布流程。请比较正文，看看它是否转向了 `release` 约定。
最后创建空白 Scope，观察一个完全没有可用历史的请求。应用应该保留这个空结果，继续处理用户问题。


In [ ]:
release_context = await client.prepare_context(
    PrepareContextRequest(scope_id=scope_id, query="release", max_bytes=2500)
)
assert release_context.status == "ready" and release_context.content is not None
release_items = context_items(release_context.content)
assert any("release" in item["content"].lower() or "changelog" in item["content"].lower() for item in release_items), (
    "query=release 时应选中发布约定。"
)
print(release_context.content)
blank = await client.create_scope(
    CreateScopeRequest(title="空白项目", summary="观察无历史行为", idempotency_key=f"{lab.run_id}:blank")
)
empty = await client.prepare_context(PrepareContextRequest(scope_id=blank.scope_id, query="amount", max_bytes=2500))
assert empty.status == "empty" and empty.content is None and empty.content_bytes == 0
show({"空白项目状态": empty.status, "内容": empty.content})

## 5. 放到一次模型请求的正确位置

PreparedContext 包含历史材料。接入模型时，需要让它清楚地知道这些内容的角色，并继续遵循当前指令。
下面只构造消息，不调用模型；第 10 篇会检查真正送入模型的消息。

请留意：准备材料成功，只证明 Server 找到了内容；模型是否收到、是否使用、任务是否因此完成，还要分别观察。


In [ ]:
messages = [
    {"role": "system", "content": "请协助开发订单导入器。下面是可核查的历史材料，不能代替当前指令和实时验证。"},
    {"role": "system", "content": prepared.content},
    {"role": "user", "content": question},
]
table([{"角色": message["role"], "内容预览": message["content"][:160]} for message in messages])
after = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
assert [entry.citation for entry in after.entries] == [entry.citation for entry in entries.entries]
print("准备上下文后，持久化 Memory 条目及其版本没有改变。")

## 轮到你：给另一种问题分配预算

把 `my_query` 改成 `documentation` 或某个金额主题，再调整预算。读一下完整输出，
判断哪些材料足够支持回答，哪些问题仍需要查询实时状态。这道练习没有固定的最佳预算。


In [ ]:
my_query = "documentation"
my_budget = 1800
my_context = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query=my_query, max_bytes=my_budget))
assert my_context.content_bytes <= my_budget
if my_context.status == "ready":
    assert my_context.content is not None
    assert any("documentation" in item["content"].lower() for item in context_items(my_context.content))
print(my_context.content or "本次没有可提供的历史材料。")

## 带着结果离开

你已经检查了问题到模型输入之间的一步：根据当前 query 选择材料，在共享字节预算内返回临时视图，同时保留可核查的引用。

下面关闭本篇的 Client 和 Server。实验文件仍留在教程的 `.powercontext/` 子目录，便于检查；
清理方法见 [README](README.md#清理实验数据)。如果在中途停止，请运行这个单元格，或关闭 Kernel。

下一篇：[工作做到一半，交给另一位接手](05_work_handoff.ipynb)。


In [ ]:
await lab.close()
print("本篇 Server 已关闭。")